In [23]:
!pip install --upgrade transformers==4.44.2
!pip install --upgrade peft==0.11.1
!pip install --upgrade sentence-transformers
!pip install accelerate


In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import re
import pandas as pd
import ast

In [2]:

model_name = "roberta-large-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

sem_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of the model checkpoint at roberta-large-mnli were not used when initializing R

In [3]:

def nli_entailment_prob(premise, hypothesis, max_len=512):
    enc = tokenizer(premise, hypothesis, truncation=True, max_length=max_len, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
    # Assuming label order: contradiction, neutral, entailment (common for MNLI)
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return {"p_contradiction": float(probs[0]), "p_neutral": float(probs[1]), "p_entailment": float(probs[2])}

def nli_per_chunk(answer_gt, retrieved_chunks, entail_thresh=0.7):
    scores = []
    for i, ch in enumerate(retrieved_chunks):
        s = nli_entailment_prob(premise=ch, hypothesis=answer_gt)
        s["chunk_idx"] = i
        scores.append(s)
    # Aggregate: max entailment across chunks
    best = max(scores, key=lambda d: d["p_entailment"])
    entail_any = 1 if best["p_entailment"] >= entail_thresh else 0
    return entail_any, best, scores




# Semantic AIC (non-exact match)


from sentence_transformers import SentenceTransformer, util

sem_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def semantic_aic(answer_gt, retrieved_chunks, thresh=0.75):
    a_emb = sem_model.encode([answer_gt], convert_to_tensor=True, normalize_embeddings=True)
    c_emb = sem_model.encode(retrieved_chunks, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(a_emb, c_emb)[0].cpu().tolist()
    max_sim = max(sims) if sims else 0.0
    aic_any = 1 if max_sim >= thresh else 0
    return aic_any, max_sim, sims



##Open-Book QA → F1 vs Ground Truth (extractive example with deepset/roberta-base-squad2)




qa = pipeline("question-answering", model="deepset/roberta-base-squad2")

def normalize_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def f1_score(a_pred, a_true):
    p = normalize_text(a_pred).split()
    t = normalize_text(a_true).split()
    common = set(p) & set(t)
    if len(p) == 0 or len(t) == 0:
        return 1.0 if p == t else 0.0
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(p)
    recall = len(common) / len(t)
    return 2 * precision * recall / (precision + recall)

def open_book_f1(question, retrieved_chunks, answer_gt, f1_thresh=0.7):
    # Concatenate context or iterate per-chunk and take best.
    concat_ctx = "\n\n".join(retrieved_chunks)[:2000]  # truncate for safety
    out = qa(question=question, context=concat_ctx)
    f1 = f1_score(out.get("answer", ""), answer_gt)
    answerable = 1 if f1 >= f1_thresh else 0
    return answerable, f1, out


In [4]:


samples = [
    # 1) Exact span present (should be easy AIC + NLI entailment + Answerable)
    {
        "id": "ex_01_exact_span",
        "question": "Who wrote Pride and Prejudice?",
        "answer_gt": "Jane Austen",
        "retrieved_chunks": [
            "Pride and Prejudice is a novel of manners written by Jane Austen in 1813.",
            "The Brontë sisters were also prominent novelists in the 19th century.",
            "The book explores the issues of marriage, morality, and misconceptions.",
            "Another classic is Wuthering Heights by Emily Brontë.",
            "There have been numerous film adaptations over the decades."
        ]
    },

    # 2) Paraphrase support (no exact string, but clear entailment)
    {
        "id": "ex_02_paraphrase",
        "question": "What is the capital city of Japan?",
        "answer_gt": "Tokyo",
        "retrieved_chunks": [
            "Japan's seat of government is located in its largest metropolitan area.",
            "The city that hosts the Imperial Palace serves as the political and economic center of the country.",
            "Kyoto was the former imperial capital for many centuries.",
            "The Kanto region includes the nation's primary hub for business and governance.",
            "The capital moved from Kyoto during the Meiji Restoration."
        ]
        # Note: No literal "Tokyo", but multiple hints pointing to Tokyo; tests semantic AIC + NLI.
    },

    # 3) Contradiction present (top chunk contradicts; later chunk supports)
    {
        "id": "ex_03_contradiction",
        "question": "What is the boiling point of water at sea level in Celsius?",
        "answer_gt": "100°C",
        "retrieved_chunks": [
            "At standard atmospheric pressure, pure water boils at 90°C.",  # WRONG (contradiction)
            "Boiling point depends on pressure; at high altitudes it's lower than at sea level.",
            "Under standard sea-level conditions (1 atm), liquid water transitions to gas at one hundred degrees Celsius.",
            "In scientific notation, temperatures may be expressed in Kelvin.",
            "Impurities and dissolved solutes can alter boiling temperatures."
        ]
    },

    # 4) Not enough info (none of the chunks support the answer)
    {
        "id": "ex_04_not_enough_info",
        "question": "Which protocol does HTTP use by default for transport security?",
        "answer_gt": "TLS",
        "retrieved_chunks": [
            "HTTP defines methods like GET, POST, PUT, and DELETE.",
            "Status codes range from informational to server errors (1xx–5xx).",
            "Headers allow clients and servers to exchange metadata.",
            "Caching and content negotiation help optimize web performance.",
            "HTTP/2 introduced multiplexing and header compression."
        ]
        # No mention of TLS/SSL/HTTPS; should yield AIC=0, Entail=0, likely low F1.
    },

    # 5) Numeric + unit normalization (tests exact vs normalized matching)
    {
        "id": "ex_05_numeric_units",
        "question": "What is the gravitational acceleration on Earth?",
        "answer_gt": "9.81 m/s^2",
        "retrieved_chunks": [
            "Near Earth's surface, gravitational acceleration averages about 9.8 meters per second squared.",
            "The value can slightly vary with latitude and elevation.",
            "On the Moon, gravity is roughly one-sixth of Earth's.",
            "The SI unit for acceleration is m·s^-2.",
            "In physics problems, g is often approximated as 9.8."
        ]
        # Should semantically entail; span may differ (9.8 vs 9.81).
    },

    # 6) Multi-fact justification (requires combining two details; per-chunk NLI may pass via one)
    {
        "id": "ex_06_multi_fact",
        "question": "Which company acquired Instagram and in what year?",
        "answer_gt": "Meta (Facebook), 2012",
        "retrieved_chunks": [
            "Instagram began as a photo-sharing app that quickly gained popularity.",
            "The acquisition by Facebook was announced for roughly one billion dollars.",
            "Meta is the parent company name adopted by Facebook in 2021.",
            "The deal for the Instagram acquisition closed in 2012.",
            "The app later introduced Stories and Reels to compete with other platforms."
        ]
        # Evidence is spread (who: Facebook/Meta; year: 2012). NLI may entail; QA-F1 should work.
    }
]



df = pd.DataFrame(samples)

df.to_csv("my-test-data.csv",index = False)


In [5]:


def evaluate_sample(sample):
    q = sample["question"]
    a = sample["answer_gt"]
    chunks = sample["retrieved_chunks"]  # top-5

    # Semantic AIC
    aic_any, max_sim, sims = semantic_aic(a, chunks, thresh=0.75)

    # NLI justification
    entail_any, best_entail, entail_scores = nli_per_chunk(a, chunks, entail_thresh=0.7)

    # Open-book answerability
    answerable, f1, qa_out = open_book_f1(q, chunks, a, f1_thresh=0.7)

    return {
        "AIC_any@5": aic_any,
        "max_sim": max_sim,
        "sims_per_chunk": sims,
        "Entailment@5": entail_any,
        "entail_scores" : entail_scores,
        "best_entail_chunk_idx": best_entail["chunk_idx"],
        "p_entail_best": best_entail["p_entailment"],
        "p_contradiction_best": best_entail["p_contradiction"],
        "Answerable@5": answerable,
        "F1@5": f1,
        "qa_pred_answer": qa_out.get("answer", "")
    }



def calculate_retrieval_metrics(all_results, k_list=[1, 3, 5]):
    total_samples = len(all_results)
    metrics = {}
    for k in k_list:
        hits = 0
        rr_sum = 0
        for res in all_results:
            sims = res["sims_per_chunk"]
            # Rank chunks by semantic similarity (highest sim = rank 1)
            ranks = np.argsort(sims)[::-1]
            best_chunk_idx = res["best_entail_chunk_idx"]

            actual_rank = np.where(ranks == best_chunk_idx)[0][0] + 1

            if actual_rank <= k:
                hits += 1
                rr_sum += 1 / actual_rank

        metrics[f"HitRate@{k}"] = round(hits / total_samples, 4)
        metrics[f"MRR@{k}"] = round(rr_sum / total_samples, 4)
    return metrics

In [6]:
df_load = pd.read_csv("/content/my-test-data.csv")



In [7]:
a = list(df_load['retrieved_chunks'].apply(ast.literal_eval))

In [9]:


max_sim = 0
p_entail_best = 0
all_eval_results = []
for i in range(0,len(df_load)):

    row_dict = df_load.iloc[i].to_dict()

    row_dict['retrieved_chunks'] = ast.literal_eval(row_dict['retrieved_chunks'])\

    val = evaluate_sample(row_dict)

    all_eval_results.append(val)

    print(f"max_sim: {val['max_sim']:.2f} | p_entail_best: {val['p_entail_best']:.2f}")

    max_sim += val['max_sim']
    p_entail_best += val['p_entail_best']


ret_metrics = calculate_retrieval_metrics(all_eval_results)


for k, v in ret_metrics.items():

    ret_metrics[k] = float(v)



max_sim = max_sim / len(df_load)
p_entail_best = p_entail_best / len(df_load)


ret_metrics["max_sim"] = max_sim
ret_metrics["p_entail_best"] = p_entail_best


print()
print()
print(ret_metrics)
print()
print()




max_sim: 0.60 | p_entail_best: 0.92
max_sim: 0.52 | p_entail_best: 0.43
max_sim: 0.52 | p_entail_best: 0.77
max_sim: 0.21 | p_entail_best: 0.13
max_sim: 0.59 | p_entail_best: 0.14
max_sim: 0.69 | p_entail_best: 0.05


{'HitRate@1': 0.1667, 'MRR@1': 0.1667, 'HitRate@3': 0.8333, 'MRR@3': 0.4722, 'HitRate@5': 1.0, 'MRR@5': 0.5056, 'max_sim': 0.5222256183624268, 'p_entail_best': 0.40609324413041276}


